In [ ]:
# ============================================================
# CELL 0 — Install Required Packages (Run Once)
# ============================================================
# FIX: Added torch for MLP support (new extension)
!pip install --upgrade pip
!pip install xgboost shap tqdm seaborn pulp joblib pandas scikit-learn torch

In [ ]:
# ============================================================
# CELL 1 — Core Imports
# ============================================================
# FIX: Merged the stray 'import shap' cell into a proper consolidated imports cell
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import os
import warnings
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, PULP_CBC_CMD

warnings.filterwarnings('ignore')
print("All imports successful.")

In [ ]:
# ============================================================
# CELL 2 — Mount Drive and Create Experiment Folder
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

BASE_DRIVE = '/content/drive/MyDrive/sem4'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
EXP_DIR = os.path.join(BASE_DRIVE, 'experiments', f'run_{timestamp}')

os.makedirs(os.path.join(EXP_DIR, 'plots'), exist_ok=True)
os.makedirs(os.path.join(EXP_DIR, 'tables'), exist_ok=True)

BASE_DIR = Path(EXP_DIR)

print(f"Project Workspace : {BASE_DRIVE}")
print(f"Experiment Results : {EXP_DIR}")

In [ ]:
# ============================================================
# CELL 3 — Synthetic Dataset Generators (All 7)
# ============================================================
# FIX: make_imbalanced and make_highdim_sparse were cut off in original.
# Both are now fully defined here.

# 1. Linear — Baseline for coefficient recovery validation
def make_linear(n=3000, n_info=3, n_noise=7, noise_std=0.1, seed=None):
    rng = np.random.RandomState(seed)
    X_info  = rng.normal(size=(n, n_info))
    X_noise = rng.normal(size=(n, n_noise))
    X = np.hstack([X_info, X_noise])
    coefs = np.array([2.0, -1.5, 1.0] + [0.0] * n_noise)
    y = X.dot(coefs) + rng.normal(0, noise_std, size=n)
    return X, y

# 2. Correlated Clusters — Tests Gately Paradox and coalition stability
def make_correlated_clusters(n=4000, n_clusters=3, cluster_size=4, rho=0.8, noise_std=0.1, seed=None):
    rng = np.random.RandomState(seed)
    bases  = rng.normal(size=(n, n_clusters))
    extras = []
    for j in range(n_clusters):
        for _ in range(cluster_size):
            noise = rng.normal(size=n)
            extras.append(rho * bases[:, j] + np.sqrt(max(0, 1 - rho**2)) * noise)
    X = np.column_stack(extras)
    coefs = rng.uniform(1.0, 2.0, size=n_clusters)
    y = bases.dot(coefs) + rng.normal(0, noise_std, size=n)
    return X, y

# 3. Nonlinear — Tests Nucleolus stability in complex interaction spaces
def make_nonlinear(n=4000, noise_std=0.2, n_noise=5, seed=None):
    rng = np.random.RandomState(seed)
    x1 = rng.uniform(-3, 3, size=n)
    x2 = rng.normal(size=n)
    x3 = rng.normal(size=n)
    others = rng.normal(size=(n, n_noise))
    X = np.column_stack([x1, x2, x3, others])
    y = np.sin(1.7 * x1) + 0.8 * (x2**2) - 0.5 * x3 + rng.normal(0, noise_std, size=n)
    return X, y

# 4. Interaction — XOR/AND logic; stress-tests attribution in synergy settings
def make_interaction(n=3000, noise_std=0.05, n_noise=6, seed=None):
    rng = np.random.RandomState(seed)
    x1 = rng.binomial(1, 0.5, size=n)
    x2 = rng.binomial(1, 0.5, size=n)
    x3 = rng.binomial(1, 0.5, size=n)
    x4 = rng.binomial(1, 0.5, size=n)
    y  = ((x1 ^ x2) | (x3 & x4)).astype(float)
    others = rng.normal(size=(n, n_noise))
    X = np.column_stack([x1, x2, x3, x4, others])
    return X, y

# 5. Mixed — Numerical + categorical feature heterogeneity
def make_mixed(n=4000, n_cat=3, cat_levels=None, n_num=5, seed=None):
    if cat_levels is None:
        cat_levels = [3, 4, 5]
    rng = np.random.RandomState(seed)
    num  = rng.normal(size=(n, n_num))
    cats = [rng.randint(0, L, size=n) for L in cat_levels]
    df   = pd.DataFrame(num, columns=[f'num{i}' for i in range(n_num)])
    for i, c in enumerate(cats):
        df[f'cat{i}'] = c
    y = (0.6 * df['num0'] + 0.3 * df['num1']
         + sum([(df[f'cat{i}'] == 1).astype(int) * 0.5 for i in range(len(cats))])
         + rng.normal(0, 0.2, size=n))
    X = pd.get_dummies(df, columns=[f'cat{i}' for i in range(len(cats))],
                       drop_first=True).values.astype(float)
    return X, y.values

# 6. Imbalanced — Rare-event reliability test (FIX: was cut off in original)
def make_imbalanced(n=20000, pos_ratio=0.02, n_info=3, seed=None):
    rng    = np.random.RandomState(seed)
    X_info  = rng.normal(size=(n, n_info))
    X_noise = rng.normal(size=(n, 7))
    score   = X_info[:, 0] * 2.0 + X_info[:, 1] * -1.0 + rng.normal(0, 1, n)
    thresh  = np.percentile(score, 100 * (1 - pos_ratio))
    # FIX: Keep as continuous score (regression) not binary to match value_regression pipeline
    y = score
    X = np.hstack([X_info, X_noise])
    return X, y

# 7. High-Dimensional Sparse — CEqL/CCRA noise suppression test
# (FIX: was referenced but never defined in original)
def make_highdim_sparse(n=3000, p=500, n_info=8, seed=None):
    rng  = np.random.RandomState(seed)
    informative = rng.normal(size=(n, n_info))
    noise       = rng.normal(size=(n, p - n_info))
    X    = np.hstack([informative, noise])
    coefs = np.zeros(p)
    coefs[:n_info] = rng.uniform(1.0, 2.0, size=n_info)
    y = X.dot(coefs) + rng.normal(0, 0.5, size=n)
    return X, y

print("All 7 synthetic dataset generators ready.")

In [ ]:
# ============================================================
# CELL 4 — CGT Attribution Methods
# ============================================================

def value_regression(model, x):
    """Standard value function for regression: returns the predicted scalar."""
    return float(model.predict(x.reshape(1, -1))[0])

def mask_instance_mean(x, keep_idxs, feature_means):
    """Masks features not in keep_idxs using their training means."""
    x2 = x.copy().astype(float)
    keep_set = set(keep_idxs)
    for j in range(len(x2)):
        if j not in keep_set:
            x2[j] = feature_means[j]
    return x2

# ------------------------------------------------------------------
# 1. Shapley Value (Monte Carlo Approximation)
# ------------------------------------------------------------------
def shapley_mc_instance(x, model, value_fn, feature_means,
                         n_perm=300, seed=None):
    rng  = np.random.RandomState(seed)
    n    = len(x)
    vals = np.zeros(n)
    for _ in range(n_perm):
        perm     = rng.permutation(n)
        prev_val = 0.0
        current  = []
        for idx in perm:
            new_coal = current + [int(idx)]
            v_new    = value_fn(model, mask_instance_mean(x, new_coal, feature_means))
            vals[int(idx)] += (v_new - prev_val)
            prev_val = v_new
            current  = new_coal
    return vals / n_perm

# ------------------------------------------------------------------
# 2. Banzhaf Index (Monte Carlo)
# ------------------------------------------------------------------
def banzhaf_mc_instance(x, model, value_fn, feature_means,
                          n_samples=300, seed=None):
    rng  = np.random.RandomState(seed)
    n    = len(x)
    vals = np.zeros(n)
    counts = np.zeros(n)  # FIX: track how many times each feature was evaluated
    for _ in range(n_samples):
        mask = rng.rand(n) < 0.5
        S    = [i for i in range(n) if mask[i]]
        vS   = value_fn(model, mask_instance_mean(x, S, feature_means))
        for i in range(n):
            if mask[i]:
                continue
            vSw = value_fn(model, mask_instance_mean(x, S + [i], feature_means))
            vals[i]   += (vSw - vS)
            counts[i] += 1
    # FIX: divide by actual count per feature, not global n_samples
    counts = np.where(counts == 0, 1, counts)
    return vals / counts

# ------------------------------------------------------------------
# 3. Nucleolus (LP-based Stability Attribution)
# FIX: Added individual rationality constraints (each feature >= solo value)
# ------------------------------------------------------------------
def nucleolus_instance(x, model, value_fn, feature_means):
    n               = len(x)
    feature_indices = list(range(n))
    v_map = {}
    for r in range(n + 1):
        for subset in itertools.combinations(feature_indices, r):
            v_map[subset] = value_fn(
                model, mask_instance_mean(x, list(subset), feature_means))

    prob    = LpProblem("Nucleolus", LpMinimize)
    phi     = [LpVariable(f"phi_{i}", lowBound=0) for i in range(n)]
    epsilon = LpVariable("epsilon")
    prob   += epsilon

    # Efficiency: sum of attributions = grand coalition value
    prob += lpSum(phi) == v_map[tuple(feature_indices)]

    # FIX: Individual rationality — each feature gets at least its solo value
    for i in range(n):
        prob += phi[i] >= v_map[(i,)]

    # Minimise max excess over all proper coalitions
    for r in range(1, n):
        for S in itertools.combinations(range(n), r):
            prob += v_map[S] - lpSum([phi[i] for i in S]) <= epsilon

    prob.solve(PULP_CBC_CMD(msg=0))

    result = np.array([phi[i].varValue if phi[i].varValue is not None else 0.0
                       for i in range(n)])
    return result

# ------------------------------------------------------------------
# 4. Gately Point (Bargaining-based)
# FIX: Added NaN/inf guard for the Gately Paradox (redundant feature collapse)
# ------------------------------------------------------------------
def gately_instance(x, model, value_fn, feature_means):
    n       = len(x)
    vN      = value_fn(model, x)
    vN_minus = np.array([
        value_fn(model, mask_instance_mean(x, [j for j in range(n) if j != i], feature_means))
        for i in range(n)
    ])
    singles = np.array([
        value_fn(model, mask_instance_mean(x, [i], feature_means))
        for i in range(n)
    ])

    numer = np.sum(vN - vN_minus)
    denom = np.sum(singles)

    # FIX: Gately Paradox fallback — if solo values collapse, fall back to Shapley-like equal split
    if abs(denom) < 1e-10:
        # All features have near-zero solo value (perfectly redundant) — split equally
        return np.full(n, vN / n)

    d     = numer / denom
    denom2 = 1.0 + d
    if abs(denom2) < 1e-10:
        return np.full(n, vN / n)

    alloc = np.array([
        (vN - vN_minus[i] + d * singles[i]) / denom2
        for i in range(n)
    ])

    # FIX: Catch any remaining NaN/inf from edge cases
    if not np.all(np.isfinite(alloc)):
        return np.full(n, vN / n)

    # Efficiency correction — ensure allocations sum to vN
    diff = vN - alloc.sum()
    if abs(diff) > 1e-8:
        alloc += diff / n
    return alloc

# ------------------------------------------------------------------
# 5. Constrained Equal Loss (CEqL)
# ------------------------------------------------------------------
def ceql_instance(x, model, value_fn, feature_means):
    n       = len(x)
    singles = np.array([
        value_fn(model, mask_instance_mean(x, [i], feature_means))
        for i in range(n)
    ])
    vN = value_fn(model, x)
    lo, hi = -1e6, float(np.max(singles))
    for _ in range(200):
        mid   = 0.5 * (lo + hi)
        alloc = np.maximum(singles - mid, 0.0)
        if alloc.sum() > vN:
            lo = mid
        else:
            hi = mid
    alloc = np.maximum(singles - mid, 0.0)
    diff  = vN - alloc.sum()
    if abs(diff) > 1e-8:
        pos = alloc > 0
        if pos.sum() > 0:
            alloc[pos] += diff / pos.sum()
    return alloc

# ------------------------------------------------------------------
# 6. Conflicting Claims with Random Arrival (CCRA)
# ------------------------------------------------------------------
def ccra_instance(x, model, value_fn, feature_means,
                   n_perms=300, seed=None):
    rng     = np.random.RandomState(seed)
    n       = len(x)
    singles = np.array([
        value_fn(model, mask_instance_mean(x, [i], feature_means))
        for i in range(n)
    ])
    vN     = value_fn(model, x)
    allocs = np.zeros(n)
    for _ in range(n_perms):
        perm      = rng.permutation(n)
        remaining = vN
        local     = np.zeros(n)
        for idx in perm:
            take        = min(singles[idx], remaining)
            local[idx]  = take
            remaining  -= take
            if remaining <= 1e-12:
                break
        allocs += local
    return allocs / n_perms

# ------------------------------------------------------------------
# 7. Shapley-Shubik Index (Threshold-based Power)
# ------------------------------------------------------------------
def shapley_shubik_instance(x, model, value_fn, feature_means,
                              threshold=0.9, n_perm=300, seed=None):
    rng        = np.random.RandomState(seed)
    n          = len(x)
    vN         = value_fn(model, x)
    target_val = threshold * vN
    vals       = np.zeros(n)
    for _ in range(n_perm):
        perm        = list(rng.permutation(n))
        current_val = 0.0
        for pos, idx in enumerate(perm):
            v_new = value_fn(
                model,
                mask_instance_mean(x, [int(i) for i in perm[:pos + 1]], feature_means)
            )
            if current_val < target_val and v_new >= target_val:
                vals[int(idx)] += 1
                break
            current_val = v_new
    return vals / n_perm

print("All 7 CGT attribution methods ready.")

In [ ]:
# ============================================================
# CELL 5 — Evaluation Metrics
# ============================================================
# FIX: sufficiency_at_k direction corrected — now measures how much
# top-k features recover the prediction above the all-masked baseline.
# Higher = better (top-k features are sufficient to explain the model).

def infidelity_metric(model, x, phi, delta_sampler,
                       num_samples=100, value_fn=value_regression):
    """Lower is better. Measures gap between attribution and model response."""
    errs = []
    f_x  = value_fn(model, x)
    for _ in range(num_samples):
        delta = delta_sampler(x.shape)
        f_xd  = value_fn(model, x - delta)
        errs.append((f_x - f_xd - np.dot(phi, delta)) ** 2)
    return float(np.mean(errs))

def sensitivity_at_k(model, x, phi, feature_means,
                      k=3, value_fn=value_regression):
    """Lower is better. Stability under top-k feature removal."""
    base = value_fn(model, x)
    topk = np.argsort(-np.abs(phi))[:k]
    x_abl = x.copy()
    for j in topk:
        x_abl[j] = feature_means[j]
    return abs(base - value_fn(model, x_abl))

def sufficiency_at_k(model, x, phi, feature_means,
                      k=3, value_fn=value_regression):
    """
    Higher is better.
    FIX: Measures how much top-k features recover prediction above
    the all-masked baseline (not gap from full prediction).
    """
    # Baseline: all features masked (mean prediction)
    x_baseline = np.array(feature_means, dtype=float)
    base_pred  = value_fn(model, x_baseline)

    # Top-k only prediction
    topk   = np.argsort(-np.abs(phi))[:k]
    x_topk = x_baseline.copy()
    for j in topk:
        x_topk[j] = x[j]
    topk_pred = value_fn(model, x_topk)

    # Full prediction
    full_pred = value_fn(model, x)

    # Sufficiency = how much of the full prediction gap does top-k recover?
    total_gap = abs(full_pred - base_pred)
    if total_gap < 1e-10:
        return 1.0  # model is flat — all methods are equally sufficient
    recovered = abs(topk_pred - base_pred)
    return float(np.clip(recovered / total_gap, 0.0, 1.0))

def sparsity_metric(phi, threshold=1e-4):
    """Higher is better. Measures conciseness of the explanation."""
    n        = len(phi)
    non_zero = np.sum(np.abs(phi) > threshold)
    return 1.0 - (non_zero / n)

print("Evaluation metrics (Infidelity, Sensitivity, Sufficiency, Sparsity) ready.")

In [ ]:
# ============================================================
# CELL 6 — SHAP Output Normalizer
# ============================================================

def _normalize_shap_output(sv, X_eval, X_train_s):
    """Normalises SHAP output to shape (n_eval, d) for all explainer types."""
    if isinstance(sv, (list, tuple)):
        try:
            sv = np.array(sv)
        except Exception:
            sv = np.stack([np.asarray(x) for x in sv], axis=0)

    sv     = np.asarray(sv)
    n_eval = X_eval.shape[0]
    d      = X_train_s.shape[1]

    # Case 1: Regression (n_eval, d)
    if sv.ndim == 2 and sv.shape[0] == n_eval and sv.shape[1] == d:
        return sv

    # Case 2: Classification (n_classes, n_eval, d)
    if sv.ndim == 3 and sv.shape[1] == n_eval and sv.shape[2] == d:
        return sv[1] if sv.shape[0] == 2 else sv[0]

    # Case 3: Reshape fallback
    if sv.size == n_eval * d:
        return sv.reshape(n_eval, d)

    raise ValueError(
        f"Unrecognised SHAP output shape {sv.shape} for n_eval={n_eval}, d={d}.")

print("SHAP normalizer ready.")

In [ ]:
# ============================================================
# CELL 7 — Experiment Engine (run_experiment)
# ============================================================
# FIX 1: Added MLP (MLPRegressor) as model_type='mlp'
# FIX 2: delta_scale raised from 0.01 to 0.1 for meaningful infidelity
# FIX 3: Global random seed set at start of each repeat for reproducibility

def run_experiment(model_type='rf', dataset_fn=None, dataset_kwargs=None,
                   n_repeats=3, n_eval=30,
                   shapley_samples=300, banzhaf_samples=300, ccra_perms=200,
                   infid_delta_samples=50,
                   delta_scale=0.1,          # FIX: was 0.01, now 0.1
                   k_top=3,
                   include_nucleolus=True, nucleolus_max_enum=12,
                   use_kernel_for_shap=True, kernel_nsamples=None,
                   save_prefix='exp'):

    if dataset_kwargs is None:
        dataset_kwargs = {}
    all_summaries = []

    for rep in range(n_repeats):
        seed = 42 + rep
        # FIX: Set global numpy seed for full reproducibility per repeat
        np.random.seed(seed)

        X, y = dataset_fn(seed=seed, **dataset_kwargs)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=seed)

        scaler        = StandardScaler().fit(X_train)
        X_train_s     = scaler.transform(X_train)
        X_test_s      = scaler.transform(X_test)
        feature_means = X_train_s.mean(axis=0)
        feature_stds  = X_train_s.std(axis=0) + 1e-12
        d             = X_train_s.shape[1]

        # ----------------------------------------------------------
        # Model Initialization
        # FIX: Added MLP model type
        # ----------------------------------------------------------
        if model_type == 'linear':
            model    = LinearRegression()
            value_fn = value_regression

        elif model_type == 'rf':
            model    = RandomForestRegressor(
                n_estimators=200, random_state=seed, n_jobs=-1)
            value_fn = value_regression

        elif model_type == 'xgb':
            model    = xgb.XGBRegressor(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                n_jobs=-1, random_state=seed)
            value_fn = value_regression

        elif model_type == 'mlp':
            # FIX: New MLP model — 3-layer network, suitable for tabular data
            model    = MLPRegressor(
                hidden_layer_sizes=(128, 64, 32),
                activation='relu',
                solver='adam',
                max_iter=500,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=seed
            )
            value_fn = value_regression

        else:
            raise ValueError(
                f"model_type must be 'linear', 'rf', 'xgb', or 'mlp'. Got: {model_type}")

        model.fit(X_train_s, y_train)
        r2 = r2_score(y_test, model.predict(X_test_s))
        print(f"  [rep {rep}] {model_type} | R2: {r2:.4f} | features: {d}")

        # Evaluation subset — use seeded choice for reproducibility
        rng_eval = np.random.RandomState(seed)
        chosen   = rng_eval.choice(
            range(len(y_test)), size=min(n_eval, len(y_test)), replace=False)
        X_eval = X_test_s[chosen]

        # ----------------------------------------------------------
        # SHAP Baseline Methods
        # ----------------------------------------------------------
        shap_methods = {}

        if model_type in ['rf', 'xgb']:
            try:
                expl_tree = shap.TreeExplainer(model)
                sv_tree   = expl_tree.shap_values(X_eval)
                shap_methods['TreeExplainer'] = _normalize_shap_output(
                    sv_tree, X_eval, X_train_s)
            except Exception as e:
                print(f"    TreeExplainer failed: {e}")

        if use_kernel_for_shap:
            try:
                bg_idx = rng_eval.choice(
                    len(X_train_s), size=min(50, len(X_train_s)), replace=False)
                bg = X_train_s[bg_idx]
                ke = shap.KernelExplainer(model.predict, bg)
                ns = kernel_nsamples if kernel_nsamples is not None else min(512 + 2 * d, 2048)
                sv_kernel = ke.shap_values(X_eval, nsamples=ns)
                shap_methods['KernelExplainer'] = _normalize_shap_output(
                    sv_kernel, X_eval, X_train_s)
            except Exception as e:
                print(f"    KernelExplainer failed: {e}")

        # Linear Explainer (works for linear and MLP with linear approximation)
        if model_type == 'linear':
            try:
                expl_lin = shap.LinearExplainer(model, X_train_s)
                sv_lin   = expl_lin.shap_values(X_eval)
                shap_methods['LinearExplainer'] = _normalize_shap_output(
                    sv_lin, X_eval, X_train_s)
            except Exception as e:
                print(f"    LinearExplainer failed: {e}")

        # ----------------------------------------------------------
        # CGT Attribution Methods
        # ----------------------------------------------------------
        methods = list(shap_methods.keys()) + [
            'Shapley_MC', 'Banzhaf_MC', 'CEqL', 'CCRA', 'Gately', 'Shapley_Shubik'
        ]
        if include_nucleolus and d <= nucleolus_max_enum:
            methods.append('Nucleolus')

        attributions = {m: [] for m in methods}

        for i, x in enumerate(tqdm(X_eval, desc=f"  Rep {rep} | {model_type} | Attributions")):
            # SHAP
            for m, mat in shap_methods.items():
                attributions[m].append(mat[i])

            # CGT Methods
            attributions['Shapley_MC'].append(
                shapley_mc_instance(x, model, value_fn, feature_means,
                                    n_perm=shapley_samples, seed=seed + i))
            attributions['Banzhaf_MC'].append(
                banzhaf_mc_instance(x, model, value_fn, feature_means,
                                    n_samples=banzhaf_samples, seed=seed + i))
            attributions['CEqL'].append(
                ceql_instance(x, model, value_fn, feature_means))
            attributions['CCRA'].append(
                ccra_instance(x, model, value_fn, feature_means,
                              n_perms=ccra_perms, seed=seed + i))
            attributions['Gately'].append(
                gately_instance(x, model, value_fn, feature_means))
            attributions['Shapley_Shubik'].append(
                shapley_shubik_instance(x, model, value_fn, feature_means,
                                        threshold=0.9,
                                        n_perm=shapley_samples,
                                        seed=seed + i))
            if 'Nucleolus' in methods:
                attributions['Nucleolus'].append(
                    nucleolus_instance(x, model, value_fn, feature_means))

        # ----------------------------------------------------------
        # Metric Evaluation Loop
        # ----------------------------------------------------------
        # FIX: delta_scale now uses feature_stds properly scaled by 0.1
        delta_fn = lambda shape: np.random.normal(
            0, delta_scale * feature_stds, size=shape)

        for m in attributions:
            infs, sens, suff, spars = [], [], [], []
            for i, x in enumerate(X_eval):
                phi = np.array(attributions[m][i])
                if not np.all(np.isfinite(phi)):
                    phi = np.zeros_like(phi)  # Guard against NaN attributions
                infs.append(infidelity_metric(
                    model, x, phi, delta_fn,
                    num_samples=infid_delta_samples, value_fn=value_fn))
                sens.append(sensitivity_at_k(
                    model, x, phi, feature_means, k=k_top, value_fn=value_fn))
                suff.append(sufficiency_at_k(
                    model, x, phi, feature_means, k=k_top, value_fn=value_fn))
                spars.append(sparsity_metric(phi))

            all_summaries.append({
                'rep'       : rep,
                'method'    : m,
                'dataset'   : dataset_fn.__name__,
                'model'     : model_type,
                'r2'        : r2,
                'inf_mean'  : np.mean(infs),
                'sens_mean' : np.mean(sens),
                'suff_mean' : np.mean(suff),
                'spars_mean': np.mean(spars)
            })

    df_summary = pd.DataFrame(all_summaries)
    out_path   = os.path.join(
        EXP_DIR, f"{save_prefix}_{model_type}_summary.csv")
    df_summary.to_csv(out_path, index=False)
    return df_summary

print("Experiment engine ready. Supports: linear | rf | xgb | mlp")

In [ ]:
# ============================================================
# CELL 8 — Experiment Registry and Parameters
# ============================================================
# FIX: n_repeats raised to 3, n_eval raised to 30, delta_scale fixed to 0.1
# All parameters named as constants with justification comments

DATASETS = {
    "linear"     : (make_linear,            dict(n=3000, n_info=3,   n_noise=7,    noise_std=0.1)),
    "nonlinear"  : (make_nonlinear,          dict(n=4000, noise_std=0.2, n_noise=5)),
    "correlated" : (make_correlated_clusters, dict(n=4000, n_clusters=3, cluster_size=4, rho=0.8)),
    "interaction": (make_interaction,         dict(n=3000, noise_std=0.05, n_noise=6)),
    "mixed"      : (make_mixed,               dict(n=4000)),
    "imbalanced" : (make_imbalanced,          dict(n=20000, pos_ratio=0.02)),
    "highdim"    : (make_highdim_sparse,      dict(n=3000, p=500, n_info=8))
}

# ---------------------------------------------------------------
# Shared parameters across all model runs
# ---------------------------------------------------------------
N_REPEATS          = 3     # 3 seeds → variance estimate for metrics
N_EVAL             = 30    # 30 explained samples per dataset per repeat
SHAPLEY_SAMPLES    = 300   # MC permutations for Shapley / Shapley-Shubik
BANZHAF_SAMPLES    = 300   # MC samples for Banzhaf Index
CCRA_PERMS         = 200   # Arrival permutations for CCRA
INFID_DELTA_SAMPLES = 50   # Perturbations for Infidelity metric
DELTA_SCALE        = 0.1   # FIX: was 0.01 — now 0.1 for meaningful perturbation
K_TOP              = 3     # k for Sensitivity and Sufficiency
INCLUDE_NUCLEOLUS  = True  # Nucleolus is exact — only runs when d <= 12

COMMON_KWARGS = dict(
    n_repeats           = N_REPEATS,
    n_eval              = N_EVAL,
    shapley_samples     = SHAPLEY_SAMPLES,
    banzhaf_samples     = BANZHAF_SAMPLES,
    ccra_perms          = CCRA_PERMS,
    infid_delta_samples = INFID_DELTA_SAMPLES,
    delta_scale         = DELTA_SCALE,
    k_top               = K_TOP,
    include_nucleolus   = INCLUDE_NUCLEOLUS
)

def kernel_nsamples_for_dim(p):
    """Adaptive KernelSHAP sample count — prevents blowup in high-dim cases."""
    return min(512 + 2 * p, 2048)

print("Registry ready.")
print(f"  Repeats={N_REPEATS} | Eval samples={N_EVAL} | delta_scale={DELTA_SCALE}")

In [ ]:
# ============================================================
# CELL 9 — RUN 1: Linear Regression on All Datasets
# ============================================================

print("="*60)
print("RUN 1: LINEAR REGRESSION")
print("="*60)

all_results_linear = []

for dataset_name, (dataset_fn, dataset_kwargs) in tqdm(
        DATASETS.items(), desc="Linear Regression"):
    X_tmp, _ = dataset_fn(**dataset_kwargs, seed=0)
    p = X_tmp.shape[1]
    print(f"\n--- Dataset: {dataset_name} | features: {p} ---")

    df_res = run_experiment(
        model_type       = "linear",
        dataset_fn       = dataset_fn,
        dataset_kwargs   = dataset_kwargs,
        use_kernel_for_shap = True,
        kernel_nsamples  = kernel_nsamples_for_dim(p),
        save_prefix      = dataset_name,
        **COMMON_KWARGS
    )
    df_res["dataset"] = dataset_name
    all_results_linear.append(df_res)

df_linear_all = pd.concat(all_results_linear, ignore_index=True)
df_linear_all.to_csv(BASE_DIR / "ALL_LINEAR_RESULTS.csv", index=False)
print("\n✅ Linear Regression complete.")

In [ ]:
# ============================================================
# CELL 10 — RUN 2: Random Forest on All Datasets
# ============================================================

print("="*60)
print("RUN 2: RANDOM FOREST")
print("="*60)

all_results_rf = []

for dataset_name, (dataset_fn, dataset_kwargs) in tqdm(
        DATASETS.items(), desc="Random Forest"):
    X_tmp, _ = dataset_fn(**dataset_kwargs, seed=0)
    p = X_tmp.shape[1]
    print(f"\n--- Dataset: {dataset_name} | features: {p} ---")

    df_res = run_experiment(
        model_type       = "rf",
        dataset_fn       = dataset_fn,
        dataset_kwargs   = dataset_kwargs,
        use_kernel_for_shap = True,
        kernel_nsamples  = kernel_nsamples_for_dim(p),
        save_prefix      = dataset_name,
        **COMMON_KWARGS
    )
    df_res["dataset"] = dataset_name
    all_results_rf.append(df_res)

df_rf_all = pd.concat(all_results_rf, ignore_index=True)
df_rf_all.to_csv(BASE_DIR / "ALL_RF_RESULTS.csv", index=False)
print("\n✅ Random Forest complete.")

In [ ]:
# ============================================================
# CELL 11 — RUN 3: XGBoost on All Datasets
# ============================================================

print("="*60)
print("RUN 3: XGBOOST")
print("="*60)

all_results_xgb = []

for dataset_name, (dataset_fn, dataset_kwargs) in tqdm(
        DATASETS.items(), desc="XGBoost"):
    X_tmp, _ = dataset_fn(**dataset_kwargs, seed=0)
    p = X_tmp.shape[1]
    print(f"\n--- Dataset: {dataset_name} | features: {p} ---")

    df_res = run_experiment(
        model_type       = "xgb",
        dataset_fn       = dataset_fn,
        dataset_kwargs   = dataset_kwargs,
        use_kernel_for_shap = True,
        kernel_nsamples  = kernel_nsamples_for_dim(p),
        save_prefix      = dataset_name,
        **COMMON_KWARGS
    )
    df_res["dataset"] = dataset_name
    all_results_xgb.append(df_res)

df_xgb_all = pd.concat(all_results_xgb, ignore_index=True)
df_xgb_all.to_csv(BASE_DIR / "ALL_XGB_RESULTS.csv", index=False)
print("\n✅ XGBoost complete.")

In [ ]:
# ============================================================
# CELL 12 — RUN 4: MLP (NEW EXTENSION) on All Datasets
# ============================================================
# This is the new deep learning extension added for Sem 4.
# MLP is a true black box — no TreeExplainer available.
# Research question: Do CGT methods remain stable and faithful
# when the model has no explicit coefficients or tree splits?

print("="*60)
print("RUN 4: MLP (Neural Network — New Extension)")
print("="*60)

all_results_mlp = []

# Nucleolus is too slow for high-dim MLP runs — disable for highdim
for dataset_name, (dataset_fn, dataset_kwargs) in tqdm(
        DATASETS.items(), desc="MLP"):
    X_tmp, _ = dataset_fn(**dataset_kwargs, seed=0)
    p = X_tmp.shape[1]
    print(f"\n--- Dataset: {dataset_name} | features: {p} ---")

    # Disable Nucleolus for high-dimensional datasets (exponential cost)
    use_nucleolus = (p <= 12)

    mlp_kwargs = {**COMMON_KWARGS, 'include_nucleolus': use_nucleolus}

    df_res = run_experiment(
        model_type          = "mlp",
        dataset_fn          = dataset_fn,
        dataset_kwargs      = dataset_kwargs,
        use_kernel_for_shap = True,
        kernel_nsamples     = kernel_nsamples_for_dim(p),
        save_prefix         = dataset_name,
        **mlp_kwargs
    )
    df_res["dataset"] = dataset_name
    all_results_mlp.append(df_res)

df_mlp_all = pd.concat(all_results_mlp, ignore_index=True)
df_mlp_all.to_csv(BASE_DIR / "ALL_MLP_RESULTS.csv", index=False)
print("\n✅ MLP complete.")

In [ ]:
# ============================================================
# CELL 13 — Sanity Check: Linear Dataset Attribution Plot
# ============================================================
# Truth: X1=2.0, X2=-1.5, X3=1.0, X4-X10=0.0
# All methods should rank X1 > X3 > X2 > noise features

print("Running sanity check on linear dataset...")

X_sc, y_sc = make_linear(n=1000, n_info=3, n_noise=7, noise_std=0.01, seed=42)
feature_names_sc = [f"X{i+1}" for i in range(X_sc.shape[1])]

scaler_sc      = StandardScaler().fit(X_sc)
X_sc_s         = scaler_sc.transform(X_sc)
feature_means_sc = np.zeros(X_sc.shape[1])  # Mean is 0 after scaling

model_sc = LinearRegression()
model_sc.fit(X_sc_s, y_sc)

x_sample = X_sc_s[0]

sc_methods = {
    'Shapley_MC'   : lambda x, m, v, f: shapley_mc_instance(x, m, v, f, n_perm=300),
    'Banzhaf_MC'   : lambda x, m, v, f: banzhaf_mc_instance(x, m, v, f, n_samples=300),
    'CEqL'         : ceql_instance,
    'CCRA'         : lambda x, m, v, f: ccra_instance(x, m, v, f, n_perms=200),
    'Gately'       : gately_instance,
    'Shapley_Shubik': lambda x, m, v, f: shapley_shubik_instance(x, m, v, f, threshold=0.9),
    'Nucleolus'    : nucleolus_instance
}

# SHAP LinearExplainer
expl_lin  = shap.LinearExplainer(model_sc, X_sc_s)
shap_vals  = expl_lin.shap_values(X_sc_s[0:1])[0]

sc_results = {'SHAP_Linear': shap_vals}
for name, func in sc_methods.items():
    print(f"  Computing {name}...")
    sc_results[name] = func(x_sample, model_sc, value_regression, feature_means_sc)

# Plot — first 5 features only (X1-X3 are informative, X4-X5 are noise)
plot_data = []
for method_name, phi in sc_results.items():
    for i in range(5):
        plot_data.append({
            'Feature'   : feature_names_sc[i],
            'Importance': np.abs(phi[i]),
            'Method'    : method_name
        })

df_sc_plot = pd.DataFrame(plot_data)

plt.figure(figsize=(13, 6))
sns.barplot(x='Feature', y='Importance', hue='Method', data=df_sc_plot)
plt.title("Sanity Check: Linear Regression Attribution\n"
          "Expected: X1 > X2 ≈ X3 >> X4, X5 (noise)", fontsize=13)
plt.ylabel("Absolute Attribution Value")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(EXP_DIR, 'plots', 'sanity_check_linear.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Sanity check plot saved.")

In [ ]:
# ============================================================
# CELL 14 — Visualization: Performance Profile Bar Charts
# ============================================================
# Generates the performance profile plots for each model x dataset
# (same format as your Sem 3 presentation slides)

def plot_performance_profiles(df, model_label, save_dir):
    """
    Generates a 4-panel bar chart (Infidelity, Sensitivity,
    Sufficiency, Sparsity) for each dataset, styled like the
    presentation slides.
    """
    metrics = [
        ('inf_mean',   'Infidelity',   'Lower is Better',  'Reds_r'),
        ('sens_mean',  'Sensitivity',  'Lower is Better',  'Oranges_r'),
        ('suff_mean',  'Sufficiency',  'Higher is Better', 'Greens'),
        ('spars_mean', 'Sparsity',     'Higher is Better', 'Blues'),
    ]

    datasets = df['dataset'].unique()

    for dset in datasets:
        sub = df[df['dataset'] == dset].groupby('method').mean(numeric_only=True).reset_index()

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(
            f"Performance Profile — {model_label} | Dataset: {dset}",
            fontsize=14, fontweight='bold')

        for ax, (col, label, direction, cmap) in zip(axes, metrics):
            sorted_sub = sub.sort_values(col, ascending=(direction == 'Lower is Better'))
            colors = plt.get_cmap(cmap)(np.linspace(0.4, 0.85, len(sorted_sub)))
            ax.barh(sorted_sub['method'], sorted_sub[col], color=colors)
            ax.set_title(f"{label}\n({direction})", fontsize=11)
            ax.set_xlabel('Mean Score')
            ax.invert_yaxis()
            ax.grid(axis='x', linestyle='--', alpha=0.5)

        plt.tight_layout()
        fname = os.path.join(save_dir, f'perf_{model_label}_{dset}.png')
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {fname}")

# Generate for all 4 models
plot_save_dir = os.path.join(EXP_DIR, 'plots')

for label, df_model in [
    ('Linear',  df_linear_all),
    ('RF',      df_rf_all),
    ('XGBoost', df_xgb_all),
    ('MLP',     df_mlp_all),
]:
    print(f"\nGenerating performance profiles for {label}...")
    plot_performance_profiles(df_model, label, plot_save_dir)

print("\n✅ All performance profile plots saved.")

In [ ]:
# ============================================================
# CELL 15 — Visualization: Global Ranking Heatmap
# ============================================================
# Generates the global ranking heatmap (like your XGBoost heatmap
# in the Sem 3 presentation) for all 4 models.

def compute_global_ranking(df):
    """
    Ranks methods per dataset per metric.
    Infidelity, Sensitivity: lower rank = better (rank 1 = best).
    Sufficiency, Sparsity:   higher rank = better (rank 1 = best).
    Returns average rank across all metrics per (method, dataset).
    """
    agg = df.groupby(['method', 'dataset']).mean(numeric_only=True).reset_index()

    for dset in agg['dataset'].unique():
        mask = agg['dataset'] == dset
        sub  = agg[mask].copy()

        # Lower is better — rank ascending
        for col in ['inf_mean', 'sens_mean']:
            agg.loc[mask, f'rank_{col}'] = sub[col].rank(ascending=True)

        # Higher is better — rank descending (rank 1 = highest value)
        for col in ['suff_mean', 'spars_mean']:
            agg.loc[mask, f'rank_{col}'] = sub[col].rank(ascending=False)

    rank_cols = ['rank_inf_mean', 'rank_sens_mean',
                 'rank_suff_mean', 'rank_spars_mean']
    agg['avg_rank'] = agg[rank_cols].mean(axis=1)
    return agg

def plot_ranking_heatmap(df, model_label, save_dir):
    ranking = compute_global_ranking(df)
    pivot   = ranking.pivot(index='method', columns='dataset', values='avg_rank')
    pivot   = pivot.round(1)

    plt.figure(figsize=(max(10, len(pivot.columns) * 1.4), max(6, len(pivot) * 0.9)))
    sns.heatmap(
        pivot,
        annot=True, fmt='.1f',
        cmap='RdYlGn_r',
        linewidths=0.5,
        cbar_kws={'label': 'Avg Rank (1=Best, Higher=Worse)'}
    )
    plt.title(
        f"Global Method Ranking — {model_label}\n"
        "(1=Best, Higher=Worse | avg across Infidelity, Sensitivity, Sufficiency, Sparsity)",
        fontsize=13
    )
    plt.xlabel('Dataset')
    plt.ylabel('Attribution Method')
    plt.tight_layout()
    fname = os.path.join(save_dir, f'ranking_heatmap_{model_label}.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Heatmap saved: {fname}")

# Generate heatmaps for all 4 models
for label, df_model in [
    ('Linear',  df_linear_all),
    ('RF',      df_rf_all),
    ('XGBoost', df_xgb_all),
    ('MLP',     df_mlp_all),
]:
    print(f"\nGlobal ranking heatmap — {label}")
    plot_ranking_heatmap(df_model, label, plot_save_dir)

print("\n✅ All ranking heatmaps saved.")